# NetShield AI real CICIDS workflow
October 13, 2026. This notebook explains the reproducible release. The ready-trained bundle is included; execute training commands only when you want to regenerate it.

In [ ]:
from pathlib import Path
import json, joblib, pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'models').exists():
    raise RuntimeError('Open this notebook from the project root')
manifest = json.loads((ROOT / 'models/data_manifest.json').read_text())
manifest['raw_rows'], manifest['train_sample_rows'], manifest['test_sample_rows']

## Data preparation
setup_data.py verifies the archive before exposing extracted files. prepare_cicids.py reads every daily file in chunks, validates selected features, relabels attempted attacks, separates identical feature vectors and retains bounded per-class samples. Source IDs, IPs, ports, labels and timestamps never enter model features. This is a feature-hash split, not independent day/session validation.

In [ ]:
pd.DataFrame(manifest['files'])

In [ ]:
pd.DataFrame({'train': manifest['train_sample_labels'], 'holdout': manifest['test_sample_labels']}).fillna(0)

## Reproduce only when needed
Run in terminal: python prepare_cicids.py --data-dir data/cicids, then python train_cicids.py --tune. Stop the app first. The scripts are the authoritative training implementation; the original notebook remains under docs for historical reference.

In [ ]:
metrics = json.loads((ROOT / 'models/metrics.json').read_text())
pd.DataFrame({name: {k: m[k] for k in ['macro_f1','weighted_f1','attack_recall','false_positive_rate','batch_latency_p95_ms','rows_per_second']} for name,m in metrics['models'].items()}).T

## Rare attack classes
A score with two or three examples is weak evidence. Macro F1 gives each class equal weight; weighted F1 can hide rare-class weaknesses.

In [ ]:
pd.DataFrame(metrics['models']['random_forest']['classification_report']).T

## Hyperparameters
Two randomized combinations per model are evaluated with grouped two-fold CV on training-only tuning rows. Identical input hashes remain within one CV group. The selected configuration is fitted on the retained training sample. Holdout is used only for reporting.

In [ ]:
{name: m['search'] for name,m in metrics['models'].items()}

## Confidence and risk
Risk = round(configured severity × class confidence). DDoS severity 90 at 80 percent confidence is 72 out of 100, High. Confidence remains uncalibrated. Disagreements and anomalies create review flags separately.

In [ ]:
from backend.engine import risk
risk('DDoS', 0.8)

## Feature importance and anomaly review
Global tree importance is training-wide, not a causal row explanation. Isolation Forest is fitted on benign training rows; contamination is a policy threshold, not measured attack prevalence.

In [ ]:
metrics['global_feature_importances']['random_forest'][:8], metrics['anomaly_metrics']

## Real IP demonstration
These holdout-derived rows retain original dataset metadata. Predictions on this demonstration are not an independent new model evaluation.

In [ ]:
demo = pd.read_csv(ROOT / 'data/cicids_demo_real_ips.csv')
demo[['Src IP','Dst IP','Protocol','Timestamp','Label']].head()

## Honest boundaries
No UNSW adapter, live packet capture, future intrusion forecast, external notifications or cloud deployment is claimed. Read the change log and validation report for measured evidence.